# ACE-Net — Stage-2 Rerun (leakage-free)  ·  PERSON A

Retrains **Stage-2** (fusion + MLP discriminator) on a Colab T4 with the
group-aware split that removes the P2 component leakage (the bug that inflated
val AUC to ~0.998). Stage-1 extractors are loaded frozen.

Set Runtime → T4 GPU before running.

**You need uploaded:** `cremad_vectors.zip` on Drive (4 CREMA dirs), and the
checkpoints `stage1_visual_crema.pt`, `stage1_speech_text_crema.pt`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Clone repo

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git log --oneline -1

## Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## Get the vectors

Mount Drive and unzip **`cremad_vectors.zip`** (upload it to your Drive root first).
Zip contains the 4 CREMA dirs at top level: GENUINE_LastHalf, GENUINE_FirstHalf, FAKE_Paradigm1, FAKE_Paradigm2 -> they unzip to `data/CREMA-D/...`.

> NOTE: the zip's top dir must be `CREMA-D/` (zip the `CREMA-D` folder itself), so paths become `data/CREMA-D/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
ZIP = '/content/drive/MyDrive/cremad_vectors.zip'   # adjust if needed
DST = '/content/Baseline_Training/data'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DST)
print('unzipped under data/:')
for root,dirs,_ in os.walk(DST):
    depth = root[len(DST):].count(os.sep)
    if depth < 2: print(' ', root)

## Upload Stage-1 checkpoints

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
from google.colab import files
up = files.upload()
for name in up:
    os.replace(name, f'checkpoints/{name}')
print(os.listdir('checkpoints'))

## Verify leakage-free split

Confirms zero actor overlap between train and test (was 91/91 in the buggy
split).

In [ ]:
import sys; sys.path.insert(0,'/content/Baseline_Training')
import random
from src.data import manifests
from src.train_utils import group_aware_split
from src.config import TrainConfig
cfg=TrainConfig()
g,f=manifests.build_stage2_samples()
rng=random.Random(cfg.seed)
p1=[s for s in f if s.emotion=='crema_fake_p1']; p2=[s for s in f if s.emotion=='crema_fake_p2']
ne=min(len(p1),len(p2)); rng.shuffle(p1); rng.shuffle(p2)
fakes=p1[:ne]+p2[:ne]; rng.shuffle(fakes)
n=min(len(g),len(fakes)); rng.shuffle(g); chosen=g[:n]+fakes[:n]
tr,va,te=group_aware_split(chosen,lambda s:s.group_key,(0.8,0.1,0.1),cfg.seed)
print('genuine',len(g),'fake',len(f))
print(f'split {len(tr)}/{len(va)}/{len(te)}')
print('actor overlap train&test:',len({s.group_key for s in tr}&{s.group_key for s in te}),'(must be 0)')

## Train Stage-2 (30 epochs, T4, no time cap)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.train_stage2 --batch-size 32 --epochs 30 --early-stop 8 --num-workers 2

## Evaluate Table 4 — forgery detection by type

The honest headline. Expect AUC **below the leaky 0.998**; Cross-Identity
should score higher than Emotion-Tampering (as in the paper).

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage2

## Save the trained Stage-2 checkpoint back to Drive

So Person B (or the results notebook) can use it later.

In [ ]:
import shutil
shutil.copy('checkpoints/stage2_acenet.pt', '/content/drive/MyDrive/stage2_acenet.pt')
print('saved stage2_acenet.pt to Drive')